In [ ]:
from IPython.display import clear_output
!git clone --branch shantanu https://github.com/AISC-Linear-Probe-Gen/Probe-Generalisation.git
%pip install --upgrade mech-interp-toolkit
clear_output()
import os
os.chdir("/content/Probe-Generalisation/research/obfuscated_activations")

In [ ]:
from datasets import load_dataset
from typing import cast
import torch
import einops
from pathlib import Path
from huggingface_hub import HfApi, login
from google.colab import userdata
from tqdm import tqdm

from utils.data import (
    extract_user_instruction,
)

from mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed
from mech_interp_toolkit.activation_utils import get_embeddings_dict, get_activations, concat_activations
from safetensors.torch import save_file
import warnings

warnings.filterwarnings("ignore")


set_global_seed(0)
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [ ]:
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_train"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_paths = ["suffix.pt"]          # list of .pt embedding files
save_folder_names = ["obfuscated"]    # corresponding folder names under outputs/cached_activations/
batch_size = 64
compute_baseline = True               # set to False to skip baseline activation computation and saving

assert len(suffix_paths) == len(save_folder_names), \
    "suffix_paths and save_folder_names must have the same length"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load 200 examples from the circuit_breakers_train split
dataset = load_dataset(
    dataset_name,
    split=split,
)

dataset = cast(dict, dataset)
prompts_str = [extract_user_instruction(x) for x in dataset["prompt"]]

clear_output()

# Load model, tokenizer and config (once, shared across all suffixes)
model, ch_tokenizer, config = load_model_tokenizer_config(
    model_name,
    suffix="",
    system_prompt="",
    attn_type="sdpa",  # Scaled Dot Product Attention for efficiency
)

clear_output()


# load suffix
# from rashad's eval code
def load_suffix(suffix_path: str, device: torch.device):
    if not suffix_path.endswith(".pt"):
        raise ValueError("suffix_path must be a .pt embedding file")
    suffix_emb = torch.load(suffix_path, map_location=device)
    if suffix_emb.dim() == 2:
        suffix_emb = suffix_emb.unsqueeze(0)
    return suffix_emb

In [ ]:
num_layers = config.num_hidden_layers
components = [(i, "layer_out") for i in range(num_layers)]

In [ ]:
num_layers = config.num_hidden_layers
components = [(i, "layer_out") for i in range(num_layers)]

# Maps save_folder_name -> (base_collate, new_collate)
# base_collate will be None if compute_baseline=False
all_collated = {}

for suffix_path, folder_name in zip(suffix_paths, save_folder_names):
    print(f"\n=== Processing suffix: {suffix_path} -> {folder_name} ===")

    suffix_embed = load_suffix(suffix_path=suffix_path, device=device)
    len_suffix = suffix_embed.shape[1]

    num_batches = (len(prompts_str) + batch_size - 1) // batch_size

    base_collate = [] if compute_baseline else None
    new_collate = []

    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(prompts_str))

        batch_prompts = prompts_str[start_idx:end_idx]
        current_batch_size = len(batch_prompts)

        print(
            f"  Batch {batch_idx + 1}/{num_batches} (samples {start_idx} to {end_idx - 1})"
        )

        batch_dict = ch_tokenizer(prompts=batch_prompts)
        batch_embeds_dict = get_embeddings_dict(model, batch_dict)
        batch_embeds = batch_embeds_dict["inputs_embeds"]
        batch_attn_mask = batch_embeds_dict["attention_mask"]

        # broadcast suffix
        batch_suffix = einops.repeat(
            suffix_embed,
            "dummy pos d_model -> (curr_batch dummy) pos d_model",
            curr_batch=current_batch_size,
        )

        new_embeds = torch.cat(
            [batch_embeds[:, :-5, :], batch_suffix, batch_embeds[:, -5:, :]], dim=1
        )
        attn_extension = torch.ones((current_batch_size, len_suffix))
        new_attn = torch.cat([batch_attn_mask, attn_extension], dim=1)

        new_embeds_dict = {
            "inputs_embeds": new_embeds,
            "attention_mask": new_attn,
        }

        if compute_baseline:
            base_acts = get_activations(
                model, inputs=batch_embeds_dict, layer_components=components, retain_grads=False, positions=None
            ).cpu()
            base_collate.append(base_acts)

        new_acts = get_activations(
            model, inputs=new_embeds_dict, layer_components=components, retain_grads=False, positions=None
        ).cpu()
        new_collate.append(new_acts)

    all_collated[folder_name] = (base_collate, new_collate)

print("\nDone computing activations for all suffixes.")

In [ ]:
# full_base_acts = dict(concat_activations(base_collate, pad_value=0))
# full_new_acts = dict(concat_activations(new_collate, pad_value=0))

In [ ]:
from tqdm import tqdm

if compute_baseline:
    base_save_dir = Path(f"outputs/cached_activations/baseline")
    base_save_dir.mkdir(parents=True, exist_ok=True)

for folder_name, (base_collate, new_collate) in all_collated.items():
    obf_save_dir = Path(f"outputs/cached_activations/{folder_name}")
    obf_save_dir.mkdir(parents=True, exist_ok=True)

    print(f"Saving activations for: {folder_name}")
    for i, (base_acts, new_acts) in enumerate(tqdm(zip(
        base_collate if compute_baseline else [None] * len(new_collate),
        new_collate,
    ))):
        if compute_baseline:
            torch.save(base_acts, base_save_dir / f"{i}_{split}_all_pos.pt")
        torch.save(new_acts, obf_save_dir / f"{i}_{split}_all_pos.pt")

In [ ]:
login(token=userdata.get('HF_TOKEN'))

api = HfApi()
repo_id = "AISC-Linear-Probe-Gen/obfuscated_activations"

api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

if compute_baseline:
    print("Uploading baseline activations...")
    api.upload_folder(
        folder_path=base_save_dir,
        path_in_repo="baseline",
        repo_id=repo_id,
        repo_type="dataset",
    )

# Upload each suffix's obfuscated activations to its own folder
for folder_name in save_folder_names:
    obf_save_dir = Path(f"outputs/cached_activations/{folder_name}")
    print(f"Uploading {folder_name} activations...")
    api.upload_folder(
        folder_path=obf_save_dir,
        path_in_repo=folder_name,
        repo_id=repo_id,
        repo_type="dataset",
    )